[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/himanshu231204/pytorch-mastery/blob/main/03_advanced_architectures/attention_transformers.ipynb)


# 01 . Attention and Transformers

## Concept — Self-Attention & Transformers (revision notes)

**What it is**
- A mechanism where every token in a sequence builds its new representation as a weighted average of *all* tokens' value vectors, with weights computed from query-key similarity.
- A Transformer stacks attention + feed-forward blocks, with residual connections and LayerNorm, and no recurrence or convolution.
- An *encoder* block uses bidirectional self-attention; a *decoder* block adds a causal mask (and optionally cross-attention to encoder outputs).

**Why it matters**
- Any token can talk to any other in one step (path length 1), unlike RNNs where information must survive many steps.
- It is highly parallel over the sequence, which is why it scales on GPUs.
- It is the backbone of GPT/BERT/T5 and, with patches as tokens, of Vision Transformers.

**When to use**
- Sequence modelling where long-range dependencies matter (text, audio, code, patches of images).
- Use a causal mask for next-token prediction (GPT style), no mask for encoding (BERT style).
- Avoid vanilla attention for very long sequences: cost is O(T^2) in time and memory.

**Math & intuition**
- `Attention(Q,K,V) = softmax(Q K^T / sqrt(d_k)) V`. Q = what I am looking for, K = what I offer, V = what I hand over if chosen.
- Dividing by `sqrt(d_k)` keeps the dot-product variance near 1 so softmax does not saturate (section 2).
- Multi-head attention runs `h` smaller attentions in parallel on slices of the embedding so different heads can specialise.
- Attention is permutation-equivariant: it knows nothing about order until you add positional information (section 5).

### 1. Scaled dot-product attention from scratch

**What this cell does (plain language):** we implement `softmax(QK^T/sqrt(d)) V` in four lines and check that it matches PyTorch's fused `F.scaled_dot_product_attention`. Each row of the weight matrix sums to 1: it is a probability distribution over which tokens to read from.

In [1]:
import math, torch, torch.nn as nn, torch.nn.functional as F
torch.manual_seed(0)

def attention(q, k, v, mask=None):
    # q,k,v: (..., T, d). mask: bool, True = allowed to attend
    d = q.size(-1)
    scores = q @ k.transpose(-2, -1) / math.sqrt(d)      # (..., T, T) similarity of every query with every key
    if mask is not None:
        scores = scores.masked_fill(~mask, float('-inf'))  # forbidden positions get -inf -> weight 0 after softmax
    w = scores.softmax(dim=-1)                            # each row is a distribution over keys
    return w @ v, w

q = torch.randn(2, 5, 8); k = torch.randn(2, 5, 8); v = torch.randn(2, 5, 8)
out, w = attention(q, k, v)
ref = F.scaled_dot_product_attention(q, k, v)
print("out shape:", out.shape, "| weights shape:", w.shape)
print("rows sum to 1:", torch.allclose(w.sum(-1), torch.ones(2, 5)))
print("matches PyTorch SDPA:", torch.allclose(out, ref, atol=1e-6))


out shape: torch.Size([2, 5, 8]) | weights shape: torch.Size([2, 5, 5])
rows sum to 1: True
matches PyTorch SDPA: True


### 2. Why divide by sqrt(d_k)?

**What this cell does (plain language):** for random q and k with unit variance, the dot product has variance `d`. Without scaling, large `d` pushes the softmax into a near one-hot (saturated) regime where gradients vanish. We measure the score std and the max attention weight with and without scaling.

In [2]:
torch.manual_seed(0)
for d in [4, 64, 1024]:
    q = torch.randn(1000, d); k = torch.randn(1000, d)
    raw = (q * k).sum(-1)                  # unscaled dot products of 1000 random pairs
    # one query vs 16 random keys: how peaked is softmax?
    qq = torch.randn(d); kk = torch.randn(16, d)
    peak_raw = (kk @ qq).softmax(-1).max().item()
    peak_scaled = (kk @ qq / math.sqrt(d)).softmax(-1).max().item()
    print(f"d={d:5d} | std raw={raw.std():6.2f} (~sqrt(d)={math.sqrt(d):5.1f}) | max weight raw={peak_raw:.3f} vs scaled={peak_scaled:.3f}")


d=    4 | std raw=  1.94 (~sqrt(d)=  2.0) | max weight raw=0.276 vs scaled=0.146
d=   64 | std raw=  7.90 (~sqrt(d)=  8.0) | max weight raw=0.813 vs scaled=0.249
d= 1024 | std raw= 31.64 (~sqrt(d)= 32.0) | max weight raw=0.999 vs scaled=0.237


### 3. Causal mask: no peeking at the future

**What this cell does (plain language):** a decoder predicting token t must not see tokens after t. We build a lower-triangular mask and then PROVE there is no leakage: the gradient of output position 2 with respect to the inputs at positions 3+ is exactly zero.

In [3]:
T = 5
causal = torch.tril(torch.ones(T, T, dtype=torch.bool))   # True where key position <= query position
print(causal.int())

torch.manual_seed(0)
x = torch.randn(1, T, 8, requires_grad=True)
out, w = attention(x, x, x, mask=causal)
print("\nattention weights row 2 (only first 3 non-zero):", w[0, 2].detach().numpy().round(3))

out[0, 2].sum().backward()                                 # gradient of ONLY position 2's output
print("grad norm per input position:", x.grad[0].norm(dim=-1).numpy().round(3))
print("future positions (3,4) have zero gradient:", bool((x.grad[0, 3:] == 0).all()))


tensor([[1, 0, 0, 0, 0],
        [1, 1, 0, 0, 0],
        [1, 1, 1, 0, 0],
        [1, 1, 1, 1, 0],
        [1, 1, 1, 1, 1]], dtype=torch.int32)

attention weights row 2 (only first 3 non-zero): [0.009 0.012 0.98  0.    0.   ]
grad norm per input position: [0.04  0.052 2.795 0.    0.   ]
future positions (3,4) have zero gradient: True


### 4. Multi-head self-attention module

**What this cell does (plain language):** we split the embedding into `h` heads with a reshape (no loops), run attention on all heads at once as an extra batch dimension, then concatenate and mix with an output projection. We verify the head split by comparing with PyTorch's own `nn.MultiheadAttention` after copying weights.

In [4]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, n_heads):
        super().__init__()
        assert d_model % n_heads == 0, "d_model must be divisible by n_heads"
        self.h, self.dk = n_heads, d_model // n_heads
        self.q_proj = nn.Linear(d_model, d_model)
        self.k_proj = nn.Linear(d_model, d_model)
        self.v_proj = nn.Linear(d_model, d_model)
        self.o_proj = nn.Linear(d_model, d_model)

    def split(self, x):                     # (B,T,D) -> (B,h,T,dk)
        B, T, _ = x.shape
        return x.view(B, T, self.h, self.dk).transpose(1, 2)

    def forward(self, x_q, x_kv=None, mask=None):
        x_kv = x_q if x_kv is None else x_kv          # self-attention by default, cross-attention if x_kv given
        q, k, v = self.split(self.q_proj(x_q)), self.split(self.k_proj(x_kv)), self.split(self.v_proj(x_kv))
        out, w = attention(q, k, v, mask)             # mask broadcasts over batch and heads
        B, h, T, dk = out.shape
        out = out.transpose(1, 2).reshape(B, T, h * dk)   # merge heads back
        self.last_w = w.detach()
        return self.o_proj(out)

torch.manual_seed(0)
mha = MultiHeadAttention(16, 4)
x = torch.randn(2, 6, 16)
print("output:", mha(x).shape, "| per-head weights:", mha.last_w.shape)

# cross-check against nn.MultiheadAttention by copying our weights into it
ref = nn.MultiheadAttention(16, 4, batch_first=True)
with torch.no_grad():
    ref.in_proj_weight.copy_(torch.cat([mha.q_proj.weight, mha.k_proj.weight, mha.v_proj.weight]))
    ref.in_proj_bias.copy_(torch.cat([mha.q_proj.bias, mha.k_proj.bias, mha.v_proj.bias]))
    ref.out_proj.weight.copy_(mha.o_proj.weight); ref.out_proj.bias.copy_(mha.o_proj.bias)
ref_out, _ = ref(x, x, x)
print("matches nn.MultiheadAttention:", torch.allclose(mha(x), ref_out, atol=1e-5))


output: torch.Size([2, 6, 16]) | per-head weights: torch.Size([2, 4, 6, 6])
matches nn.MultiheadAttention: True


### 5. Positional encoding and permutation equivariance

**What this cell does (plain language):** self-attention treats its input as a set: shuffling the tokens just shuffles the outputs the same way. We demonstrate that, then add sinusoidal positional encodings, after which output depends on the order. Order information has to be injected explicitly.

In [5]:
def sinusoidal_pe(T, d):
    pos = torch.arange(T).unsqueeze(1)                              # (T,1)
    freq = torch.exp(torch.arange(0, d, 2) * (-math.log(10000.0) / d))  # geometric ladder of wavelengths
    pe = torch.zeros(T, d)
    pe[:, 0::2] = torch.sin(pos * freq); pe[:, 1::2] = torch.cos(pos * freq)
    return pe

torch.manual_seed(0)
mha = MultiHeadAttention(16, 4).eval()
x = torch.randn(1, 6, 16)
perm = torch.tensor([3, 1, 5, 0, 2, 4])

with torch.no_grad():
    # no positional info: permuting the input permutes the output identically
    print("equivariant without PE:", torch.allclose(mha(x)[:, perm], mha(x[:, perm]), atol=1e-5))
    pe = sinusoidal_pe(6, 16)
    xp = x + pe
    # with PE added BEFORE shuffling tokens, order now matters if we shuffle the raw tokens and re-add PE
    print("equivariant with PE (shuffle tokens, re-add PE):",
          torch.allclose(mha(xp)[:, perm], mha(x[:, perm] + pe), atol=1e-5))
print("PE shape:", pe.shape, "| first position:", pe[0, :4].numpy().round(2))


equivariant without PE: True
equivariant with PE (shuffle tokens, re-add PE): False
PE shape: torch.Size([6, 16]) | first position: [0. 1. 0. 1.]


### 6. A pre-LayerNorm encoder block

**What this cell does (plain language):** an encoder block is just: LayerNorm -> self-attention -> residual add, then LayerNorm -> MLP -> residual add. We use the 'pre-LN' arrangement (norm before each sub-layer) because it trains more stably than the original post-LN. The shape in equals the shape out, so blocks stack freely.

In [6]:
class EncoderBlock(nn.Module):
    def __init__(self, d, h, mlp_ratio=4, p=0.0):
        super().__init__()
        self.ln1, self.ln2 = nn.LayerNorm(d), nn.LayerNorm(d)
        self.attn = MultiHeadAttention(d, h)
        self.mlp = nn.Sequential(nn.Linear(d, mlp_ratio * d), nn.GELU(), nn.Linear(mlp_ratio * d, d))
        self.drop = nn.Dropout(p)

    def forward(self, x, mask=None):
        x = x + self.drop(self.attn(self.ln1(x), mask=mask))   # residual around attention
        x = x + self.drop(self.mlp(self.ln2(x)))               # residual around MLP
        return x

torch.manual_seed(0)
enc = nn.Sequential(*[EncoderBlock(32, 4) for _ in range(3)])
x = torch.randn(2, 10, 32)
print("in:", x.shape, "-> out:", enc(x).shape)
print("params per block:", sum(p.numel() for p in EncoderBlock(32, 4).parameters()))


in: torch.Size([2, 10, 32]) -> out: torch.Size([2, 10, 32])
params per block: 12704


### 7. A decoder block: causal self-attention + cross-attention

**What this cell does (plain language):** the decoder block adds a second attention whose queries come from the decoder but whose keys/values come from the encoder output ('memory'). We run a 4-token target against a 7-token source and confirm the cross-attention weights have shape (target x source), and that changing a FUTURE target token does not change earlier outputs.

In [7]:
class DecoderBlock(nn.Module):
    def __init__(self, d, h):
        super().__init__()
        self.ln1, self.ln2, self.ln3 = nn.LayerNorm(d), nn.LayerNorm(d), nn.LayerNorm(d)
        self.self_attn = MultiHeadAttention(d, h)
        self.cross_attn = MultiHeadAttention(d, h)
        self.mlp = nn.Sequential(nn.Linear(d, 4 * d), nn.GELU(), nn.Linear(4 * d, d))

    def forward(self, y, memory, causal_mask):
        y = y + self.self_attn(self.ln1(y), mask=causal_mask)           # masked: can't see future target tokens
        y = y + self.cross_attn(self.ln2(y), x_kv=memory)               # queries=decoder, keys/values=encoder
        return y + self.mlp(self.ln3(y))

torch.manual_seed(0)
dec = DecoderBlock(32, 4).eval()
memory = torch.randn(1, 7, 32)         # encoder output (source length 7)
y = torch.randn(1, 4, 32)              # decoder input   (target length 4)
cm = torch.tril(torch.ones(4, 4, dtype=torch.bool))
with torch.no_grad():
    out = dec(y, memory, cm)
    print("decoder out:", out.shape, "| cross-attn weights (B,heads,tgt,src):", tuple(dec.cross_attn.last_w.shape))
    y2 = y.clone(); y2[:, 3] += 5.0                                     # perturb the LAST target token
    out2 = dec(y2, memory, cm)
    print("positions 0-2 unchanged by future edit:", torch.allclose(out[:, :3], out2[:, :3], atol=1e-6))
    print("position 3 changed:", not torch.allclose(out[:, 3], out2[:, 3]))


decoder out: torch.Size([1, 4, 32]) | cross-attn weights (B,heads,tgt,src): (1, 4, 4, 7)
positions 0-2 unchanged by future edit: True
position 3 changed: True


### 8. Padding masks for batches of different lengths

**What this cell does (plain language):** real batches contain padded sequences; pad tokens must not receive attention. We build a key-padding mask, combine it with the causal mask using logical AND, and check that padded keys get exactly zero weight.

In [8]:
lengths = torch.tensor([5, 3])                         # true lengths of 2 sequences, padded to T=5
T = 5
valid = torch.arange(T)[None, :] < lengths[:, None]    # (B,T) True for real tokens
pad_mask = valid[:, None, None, :]                     # (B,1,1,T): broadcast over heads and queries
causal = torch.tril(torch.ones(T, T, dtype=torch.bool))
mask = pad_mask & causal                               # allowed = real key AND not in the future

torch.manual_seed(0)
mha = MultiHeadAttention(16, 4).eval()
x = torch.randn(2, T, 16)
with torch.no_grad():
    mha(x, mask=mask)
w = mha.last_w                                         # (B,h,T,T)
print("seq 1 (length 3) weights on padded keys 3,4 (should be 0):", w[1, :, :3, 3:].abs().max().item())
print("last REAL query of seq 1 attends to:", w[1, 0, 2].numpy().round(2))
print("warning: fully-masked rows give NaN; here every query still sees itself:", bool(torch.isfinite(w[:, :, :3]).all()))


seq 1 (length 3) weights on padded keys 3,4 (should be 0): 0.0
last REAL query of seq 1 attends to: [0.23 0.6  0.17 0.   0.  ]


### 9. Train a tiny character-level GPT

**What this cell does (plain language):** we generate a synthetic corpus from a small grammar, build a 2-layer decoder-only Transformer (causal self-attention only) with token + learned position embeddings, and train it for next-character prediction. The loss should fall well below the uniform-guess baseline `ln(vocab)`.

In [9]:
import random, time
random.seed(0); torch.manual_seed(0)
subj = ["the cat", "the dog", "a bird", "the fox"]; verb = ["sat", "ran", "hid", "slept"]; place = ["on the mat", "in the fog", "by the log", "at the inn"]
text = " ".join(f"{random.choice(subj)} {random.choice(verb)} {random.choice(place)}." for _ in range(400))
chars = sorted(set(text)); stoi = {c: i for i, c in enumerate(chars)}; itos = {i: c for c, i in stoi.items()}
data = torch.tensor([stoi[c] for c in text])
V, BLOCK = len(chars), 32
print("corpus chars:", len(text), "| vocab:", V, "| uniform baseline loss ln(V) =", round(math.log(V), 3))

class TinyGPT(nn.Module):
    def __init__(self, V, d=64, h=4, L=2, T=BLOCK):
        super().__init__()
        self.tok, self.pos = nn.Embedding(V, d), nn.Embedding(T, d)
        self.blocks = nn.ModuleList([EncoderBlock(d, h) for _ in range(L)])   # same block; causal mask makes it a decoder
        self.ln, self.head = nn.LayerNorm(d), nn.Linear(d, V)
        self.register_buffer("mask", torch.tril(torch.ones(T, T, dtype=torch.bool)))
    def forward(self, idx):
        B, T = idx.shape
        x = self.tok(idx) + self.pos(torch.arange(T))
        for b in self.blocks: x = b(x, mask=self.mask[:T, :T])
        return self.head(self.ln(x))

def get_batch(bs=32):
    ix = torch.randint(0, len(data) - BLOCK - 1, (bs,))
    x = torch.stack([data[i:i + BLOCK] for i in ix]); y = torch.stack([data[i + 1:i + BLOCK + 1] for i in ix])  # y = x shifted by 1
    return x, y

model = TinyGPT(V); opt = torch.optim.AdamW(model.parameters(), lr=3e-3)
t0 = time.time()
for step in range(300):
    x, y = get_batch()
    loss = F.cross_entropy(model(x).reshape(-1, V), y.reshape(-1))
    opt.zero_grad(); loss.backward(); opt.step()
    if step % 75 == 0 or step == 299: print(f"step {step:3d} | loss {loss.item():.3f}")
print(f"params: {sum(p.numel() for p in model.parameters()):,} | train time {time.time()-t0:.1f}s")


corpus chars: 9687 | vocab: 21 | uniform baseline loss ln(V) = 3.045


step   0 | loss 3.372


step  75 | loss 0.509


step 150 | loss 0.237


step 225 | loss 0.245


step 299 | loss 0.237
params: 104,853 | train time 11.1s


### 10. Generate text autoregressively

**What this cell does (plain language):** we sample one character at a time: feed the context, take the last position's logits, apply a temperature, sample, append, repeat. Low temperature is greedy/safe, high temperature is random. The model has learned the grammar's word and sentence shapes.

In [10]:
@torch.no_grad()
def generate(model, start="the ", n=80, temperature=1.0):
    model.eval()
    idx = torch.tensor([[stoi[c] for c in start]])
    for _ in range(n):
        logits = model(idx[:, -BLOCK:])[:, -1] / temperature       # only the last position predicts the next char
        nxt = torch.multinomial(logits.softmax(-1), 1)
        idx = torch.cat([idx, nxt], dim=1)
    return "".join(itos[i] for i in idx[0].tolist())

torch.manual_seed(1)
print("T=0.5:", generate(model, temperature=0.5))
print("T=1.5:", generate(model, temperature=1.5))


T=0.5: the dog sat by the log. the dog hid by the log. the fox slept by the log. the cat sl


T=1.5: the log. the a slept at the inn. a bird slept at the inn. a bird hid by the log. the


### 11. Inspect what attention learned

**What this cell does (plain language):** we feed one sentence through the trained model and plot the first layer's attention weights for one head. A lower-triangular pattern is guaranteed by the mask; the interesting part is which earlier characters each position focuses on.

In [11]:
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
s = "the cat sat on the mat."
idx = torch.tensor([[stoi[c] for c in s]])
model.eval()
with torch.no_grad(): model(idx)
w = model.blocks[0].attn.last_w[0, 0]                     # layer 0, head 0: (T,T)
fig, ax = plt.subplots(figsize=(5, 4.5))
ax.imshow(w, cmap="viridis"); ax.set_xticks(range(len(s))); ax.set_xticklabels(list(s)); ax.set_yticks(range(len(s))); ax.set_yticklabels(list(s))
ax.set_xlabel("key (attended to)"); ax.set_ylabel("query"); ax.set_title("Layer 0 / head 0 attention")
plt.tight_layout(); plt.show()
print("upper triangle is exactly zero:", bool((w.triu(1) == 0).all()))


upper triangle is exactly zero: True


## Common bugs

- **Forgetting the mask in a decoder** — the model sees the answer and training loss collapses suspiciously fast, then generation is garbage. Fix: pass a causal mask and test with the gradient check from section 3.
- **Masking with 0 instead of -inf** — `masked_fill(~mask, 0)` before softmax still gives those positions weight. Fix: fill scores with `-inf` (or a large negative) BEFORE softmax.
- **Fully masked rows give NaN** — if a query has no allowed key, softmax of all `-inf` is NaN and it poisons the whole batch. Fix: ensure each row keeps at least one key (e.g. attend to self) or `nan_to_num` the weights.
- **Using `.view` after `.transpose`** — raises 'view size is not compatible' because the tensor is non-contiguous. Fix: `.transpose(1, 2).reshape(...)` or `.contiguous().view(...)`.
- **Mixing up (B,T,D) and (T,B,D)** — `nn.MultiheadAttention` defaults to `batch_first=False`; passing batch-first data silently attends across the batch axis. Fix: set `batch_first=True`.
- **No positional information** — the model behaves like a bag of tokens. Fix: add sinusoidal/learned/rotary position encodings, and make sure learned positions cover your max length.
- **Forgetting `model.eval()` for generation** — dropout stays active and samples are noisier. Fix: call `model.eval()` and wrap in `torch.no_grad()`.
- **Feeding a context longer than the position table** — indexing `self.pos` beyond `block_size` errors. Fix: crop context with `idx[:, -BLOCK:]` as in `generate`.

## Exercises

**Exercise 1.** Implement attention dropout (dropout on the attention WEIGHTS) as an optional argument of `attention` and confirm it is a no-op in eval mode.

In [12]:
# Your attempt for Exercise 1 here


**Solution 1**

In [13]:
def attention_do(q, k, v, mask=None, p=0.1, training=True):
    d = q.size(-1)
    s = q @ k.transpose(-2, -1) / math.sqrt(d)
    if mask is not None: s = s.masked_fill(~mask, float('-inf'))
    w = F.dropout(s.softmax(-1), p=p, training=training)   # drop whole attention links
    return w @ v
torch.manual_seed(0)
q = torch.randn(1, 4, 8)
print("eval == no dropout:", torch.allclose(attention_do(q, q, q, training=False), attention(q, q, q)[0]))
print("train differs:", not torch.allclose(attention_do(q, q, q, training=True), attention(q, q, q)[0]))


eval == no dropout: True
train differs: True


**Exercise 2.** Count the parameters of `MultiHeadAttention(d, h)` for d=64 and show the count does NOT depend on the number of heads.

In [14]:
# Your attempt for Exercise 2 here


**Solution 2**

In [15]:
for h in [1, 4, 8]:
    n = sum(p.numel() for p in MultiHeadAttention(64, h).parameters())
    print(h, "heads ->", n, "params (= 4*(64*64+64) =", 4 * (64 * 64 + 64), ")")


1 heads -> 16640 params (= 4*(64*64+64) = 16640 )
4 heads -> 16640 params (= 4*(64*64+64) = 16640 )
8 heads -> 16640 params (= 4*(64*64+64) = 16640 )


**Exercise 3.** Replace the learned position embedding in `TinyGPT` with the sinusoidal one (as a fixed buffer) and train 100 steps. Does the loss still go down?

In [16]:
# Your attempt for Exercise 3 here


**Solution 3**

In [17]:
class TinyGPTSin(TinyGPT):
    def __init__(self, V):
        super().__init__(V)
        self.register_buffer("pe", sinusoidal_pe(BLOCK, 64))
    def forward(self, idx):
        B, T = idx.shape
        x = self.tok(idx) + self.pe[:T]
        for b in self.blocks: x = b(x, mask=self.mask[:T, :T])
        return self.head(self.ln(x))
torch.manual_seed(0)
m = TinyGPTSin(V); o = torch.optim.AdamW(m.parameters(), lr=3e-3)
for s in range(100):
    x, y = get_batch(); l = F.cross_entropy(m(x).reshape(-1, V), y.reshape(-1)); o.zero_grad(); l.backward(); o.step()
print("loss after 100 steps:", round(l.item(), 3))


loss after 100 steps: 0.273


**Exercise 4.** Show numerically that a causal model's logits at position t are unchanged if you modify characters AFTER t (use the trained `model`).

In [18]:
# Your attempt for Exercise 4 here


**Solution 4**

In [19]:
model.eval()
x, _ = get_batch(1)
x2 = x.clone(); x2[0, 20:] = (x2[0, 20:] + 1) % V        # corrupt positions 20..31
with torch.no_grad():
    a, b = model(x), model(x2)
print("logits before position 20 identical:", torch.allclose(a[:, :20], b[:, :20], atol=1e-5))
print("logits after 20 differ:", not torch.allclose(a[:, 20:], b[:, 20:]))


logits before position 20 identical: True
logits after 20 differ: True


**Exercise 5.** Add a KV-cache-free sanity check: compute perplexity (exp of cross-entropy) of the trained model on a fresh batch and compare it with the uniform baseline `V`.

In [20]:
# Your attempt for Exercise 5 here


**Solution 5**

In [21]:
with torch.no_grad():
    x, y = get_batch(64)
    ce = F.cross_entropy(model(x).reshape(-1, V), y.reshape(-1))
print(f"perplexity {ce.exp():.2f} vs uniform baseline {V}")


perplexity 1.26 vs uniform baseline 21


**Exercise 6.** Implement top-k sampling in `generate` (keep only the k largest logits before softmax).

In [22]:
# Your attempt for Exercise 6 here


**Solution 6**

In [23]:
@torch.no_grad()
def generate_topk(model, start="the ", n=60, k=3):
    model.eval(); idx = torch.tensor([[stoi[c] for c in start]])
    for _ in range(n):
        logits = model(idx[:, -BLOCK:])[:, -1]
        kth = logits.topk(k).values[:, -1:]
        logits = logits.masked_fill(logits < kth, float('-inf'))
        idx = torch.cat([idx, torch.multinomial(logits.softmax(-1), 1)], 1)
    return "".join(itos[i] for i in idx[0].tolist())
torch.manual_seed(0); print(generate_topk(model))


the log. the cat hid on the mat. the cat slept by the log. the d
